# 案例：工具调用与外部信息获取

> 更新时间：2026-11-13

## 核心学习目标

本案例以酒店推荐任务为对象，学习如何将地图查询能力封装为智能体可以调用的工具，通过流式执行过程，可以观察模型何时生成工具调用、工具返回哪些候选酒店，以及模型如何结合用户偏好生成最终推荐。

## 案例设计

案例首先定义 `recommend_hotel` 工具，由该工具调用百度或高德地图 POI 接口查询酒店候选列表。随后使用 `create_agent` 创建带记忆的酒店推荐 Agent。最后通过流式消息输出工具调用事件和模型回答，从而区分候选检索与最终推荐两个阶段。

地图供应商适配位于 `map_api.py`，酒店搜索与结果标准化位于 `hotel_search.py`，模型创建位于 `llm.py`。本案例需要项目根目录 `.env` 中配置 `DASHSCOPE_API_KEY`，并至少配置 `BAIDU_MAP_AK` 或 `AMAP_KEY`。

**学习重点**：
1. 理解 LangChain 工具的参数定义、返回结构和文档字符串约束。
2. 掌握使用 `create_agent` 将大模型、工具、系统提示词和 Checkpointer 组合为 Agent。
3. 观察 Agent 自动生成工具调用参数并读取工具结果的过程。
4. 区分地图候选查询与模型最终推荐两类职责。
5. 理解流式执行中模型文本、工具调用和工具结果事件的差异。

## 0. 环境

本案例使用仓库根目录的共享 `.venv`。运行 Notebook 前，请在仓库根目录执行：

```bash
uv pip install -r Chapter-5/requirements.txt
```

环境变量统一从仓库根目录 `.env` 加载，不在章节目录维护局部 `.env`。

In [ ]:
import sys
from datetime import datetime

if sys.platform == "win32":
    for stream in (sys.stdout, sys.stderr):
        try:
            stream.reconfigure(encoding="utf-8")
        except Exception:
            pass

print("当前时间:", datetime.now().astimezone().strftime("%Y-%m-%d %H:%M:%S %z"))

## 1. 依赖与模块导入

导入 Agent、工具、Checkpointer、酒店搜索模块和统一模型入口。`hotel_search.py` 负责酒店候选查询，`map_api.py` 负责底层地图请求，`llm.py` 负责读取模型相关环境变量并创建模型。

In [ ]:
import asyncio
import json

from typing import Any, Dict, Optional
from pydantic import BaseModel, Field
from langchain_core.messages import ToolMessage
from langchain_core.tools import tool
from langchain.agents import create_agent
from langgraph.checkpoint.memory import MemorySaver


# 酒店搜索模块（内部通过 map_api 调用百度 / 高德 Place API）
from hotel_search import search_hotels
from llm import create_llm

## 2. 酒店查询与结果标准化

`fetch_hotels` 负责调用 `hotel_search.py` 中的酒店搜索接口，并把百度和高德两种来源统一为 Agent 可读取的酒店字段。发生接口错误时，工具会返回错误信息而不是伪造酒店数据。

In [ ]:
def _normalize_hotel_row(item: Dict[str, Any]) -> Dict[str, Any]:
    """将地图 API 酒店结果转换为 Agent 工具输出字段。"""

    return {
        "name": item.get("name"),
        "district": item.get("district"),
        "address": item.get("address"),
        "rating": item.get("rating"),
        "avg_price": item.get("avg_price_cny") or item.get("avg_price"),
        "tel": item.get("tel"),
    }


async def fetch_hotels(city: str, keyword: str = "酒店", limit: int = 10) -> Dict[str, Any]:
    """调用地图 API 查询酒店；需配置 BAIDU_MAP_AK 或 AMAP_KEY。"""
    res = await search_hotels(city, limit=limit, keyword=keyword)
    if res.get("error"):
        return {
            "hotels": [],
            "source": "api_error",
            "error": res.get("error"),
            "message": res.get("message") or res.get("error"),
        }
    hotels = [_normalize_hotel_row(h) for h in (res.get("hotels") or []) if isinstance(h, dict)]
    return {
        "hotels": hotels,
        "source": res.get("data_source", "api"),
        "search_query": res.get("search_query"),
    }

## 3. 定义酒店推荐工具

`recommend_hotel` 使用 Pydantic 参数 Schema 和 `@tool` 声明工具能力。工具参数严格对应酒店搜索模块实际接受的输入：`city`、地图可检索的 `keyword` 和 `limit`。安静、预算等条件保留在 Agent 对话上下文中，用于读取候选后的最终选择。

In [ ]:
class HotelSearchInput(BaseModel):
    """酒店搜索 API 的实际输入参数。"""

    city: str = Field(description="目标城市，必填。")
    keyword: str | None = Field(
        default=None,
        description="地图可检索词，例如区县、景点、地标、街道或酒店品牌；不得填写安静等主观偏好。",
    )
    limit: int = Field(
        default=10,
        ge=1,
        le=20,
        description="返回酒店候选数量，范围 1 到 20。",
    )


@tool(args_schema=HotelSearchInput)
async def recommend_hotel(
    city: str,
    keyword: Optional[str] = None,
    limit: int = 10,
) -> Dict[str, Any]:
    """查询酒店候选列表，不生成最终推荐。"""

    search_key = keyword or "酒店"
    result = await fetch_hotels(city, search_key, limit)
    payload: Dict[str, Any] = {
        "city": city,
        "keyword": keyword,
        "limit": limit,
        "hotel_list": result["hotels"],
        "data_source": result["source"],
    }
    if result.get("search_query"):
        payload["search_query"] = result["search_query"]
    if result.get("error"):
        payload["error"] = result["error"]
        payload["message"] = result.get("message")
    return payload

## 4. 创建带记忆的 Agent

使用 `create_agent` 组合模型、`recommend_hotel` 工具、系统提示词和 `MemorySaver`。Checkpointer 通过 `thread_id` 维护同一会话的历史状态。

In [ ]:
# 组装带记忆的工具 Agent
llm = create_llm()
memory = MemorySaver()
agent = create_agent(
    llm,
    tools=[recommend_hotel],
    system_prompt="""你是酒店推荐助手，只能通过工具 recommend_hotel 查酒店。
规则：
1. city 必填。
2. keyword 只填写地图可检索的区县、景点、地标、街道或酒店品牌，例如“近景区”“平城区”“希尔顿”；不得填写“安静”“舒适”“性价比”等主观偏好。
3. 用户提出的安静、预算等条件保留在对话上下文中，不传给地图工具。
4. 根据工具返回的 hotel_list，结合用户全部主观偏好和预算，由你挑选并只向用户推荐一家最合适的酒店。
5. 非酒店相关问题，回复：我只能协助酒店推荐。""",
    checkpointer=memory
)

## 5. 对比：直接调用工具

本单元不经过 Agent，直接调用 `recommend_hotel` 工具。工具只接收其底层酒店搜索 API 实际支持的参数，用于查询并返回酒店候选列表，不负责生成最终推荐话术。

In [ ]:
direct_result = await recommend_hotel.ainvoke(
    {"city": "大同", "keyword": "近景区", "limit": 10}
)

print("搜索词:", direct_result.get("search_query"))
print("数据来源:", direct_result.get("data_source"))
print("候选数量:", len(direct_result.get("hotel_list", [])))

if direct_result.get("error"):
    print("查询失败:", direct_result.get("message") or direct_result["error"])
else:
    for index, hotel in enumerate(direct_result["hotel_list"], 1):
        location = hotel.get("district") or hotel.get("address")
        print(f"  {index}. {hotel.get('name')} | {location}")

## 6. Agent 自动调用工具与事件输出

`run_hotel_agent` 使用 `astream_events(version="v2")` 获取 Agent 事件流。模型自行决定何时调用 `recommend_hotel` 以及传入什么参数；函数输出模型增量文本，并在工具调用开始和结束时显示工具名称、完整调用参数与完整返回内容。

In [ ]:
# 流式执行与工具调用事件监听
def _content_to_str(content: Any) -> str:
    """将 LangChain 消息内容转换为普通文本。"""

    if isinstance(content, str):
        return content
    if isinstance(content, list):
        parts: list[str] = []
        for block in content:
            if isinstance(block, dict) and block.get("type") == "text":
                parts.append(str(block.get("text") or ""))
            elif isinstance(block, str):
                parts.append(block)
        return "".join(parts)
    return str(content) if content else ""


def _format_event_value(value: Any) -> str:
    """把工具输入或输出格式化为完整、可读的 JSON 文本。"""

    if isinstance(value, ToolMessage):
        value = value.content
    elif hasattr(value, "content"):
        value = value.content

    if isinstance(value, str):
        try:
            value = json.loads(value)
        except json.JSONDecodeError:
            return value
    return json.dumps(value, ensure_ascii=False, indent=2, default=str)


async def run_hotel_agent(user_query: str, thread_id: str = "demo_01"):
    """流式执行酒店推荐 Agent，并输出模型文本和完整工具事件。"""

    inputs = {"messages": [("user", user_query)]}
    config = {"configurable": {"thread_id": thread_id}}

    async for event in agent.astream_events(inputs, config, version="v2"):
        event_type = event["event"]
        if event_type == "on_chat_model_stream":
            chunk = event["data"].get("chunk")
            text = _content_to_str(getattr(chunk, "content", chunk))
            if text:
                yield text
        elif event_type == "on_tool_start":
            yield f"\n[调用工具：{event.get('name')}]\n"
            yield "[调用参数]\n" + _format_event_value(event["data"].get("input")) + "\n"
        elif event_type == "on_tool_end":
            yield "\n[工具返回结果]\n" + _format_event_value(event["data"].get("output")) + "\n"

## 7. 运行酒店推荐案例

默认示例问题为大同三天行程中的酒店推荐，需求包含安静和近景区两项偏好。运行后可以观察工具调用和最终推荐文本。

In [ ]:
USER_QUERY = "我要去大同玩三天给我推荐酒店，需要安静/近景区"


async def main() -> None:
    """运行 Chapter-5 默认酒店推荐示例。"""

    print(f"用户: {USER_QUERY}\n")
    print("模型回答: ", end="", flush=True)
    async for chunk in run_hotel_agent(USER_QUERY):
        print(chunk, end="", flush=True)
    print()

In [ ]:
await main()

## 小结

1. **工具定义**将酒店候选查询封装为可被模型调用的结构化能力。
2. **Agent 组装**将模型、工具、系统提示词和会话记忆连接为执行图。
3. **工具参数**需要表达城市、偏好和预算等任务条件。
4. **流式执行**可以观察工具调用开始、工具返回结果和模型最终回答。
5. **候选检索与最终推荐分离**，使地图查询和语言决策各自承担清晰职责。